# 무역 법리 LLM 파인튜닝 노트북
### Gemma-4-E2B-it · QLoRA · Unsloth · Google Colab

**환경**: Google Colab (T4 GPU 이상 권장 — E2B는 8GB VRAM에서 동작)

| 섹션 | 내용 |
|---|---|
| 0 | 환경 설정 및 드라이브 마운트 |
| 1 | 설정값 입력 |
| 2 | 데이터 로드 · Train/Val 분할 |
| 3 | 모델 로드 및 LoRA 설정 |
| 4 | 학습 실행 |
| 5 | 저장 — 어댑터 저장 (기본) / 병합 GGUF (선택) |
| 6 | 샘플 추론 검증 |

## 운용 구조
```
베이스 모델 (google/gemma-4-E2B-it)
  ├── 어댑터 OFF: OCR, 일반 용도 등 → 베이스 모델 그대로 사용
  └── 어댑터 ON:  무역 법리 판단   → 본 노트북으로 학습한 어댑터 적재
```
어댑터는 하나만 운용합니다. 용도에 따라 어댑터를 켜고 끄는 방식으로 서비스합니다.

**저장 방식 선택 기준**
- **5A (어댑터 저장, 권장)**: 베이스 모델 하나에 어댑터를 동적으로 로드/해제. OCR 등 베이스 용도와 무역 용도를 하나의 프로세스에서 전환 가능.
- **5B (병합 GGUF)**: 어댑터가 베이스에 녹아든 단일 파일. 무역 용도 전용 배포 시 사용. OCR 등 베이스 기능은 별도 프로세스 필요.

## 섹션 0. 환경 설정

In [ ]:
# GPU 확인
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 — Runtime > Change runtime type > T4 GPU 선택'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# Unsloth 설치 (Colab 전용)
import subprocess, sys

result = subprocess.run(
    ["pip", "install", "unsloth", "trl", "transformers", "datasets", "peft",
     "--quiet", "--no-deps"],
    capture_output=True, text=True
)
# Unsloth 의존성 별도 설치
subprocess.run(
    ["pip", "install", "unsloth[colab-new]", "--quiet"],
    capture_output=True, text=True
)
print("✅ Unsloth 설치 완료")

In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')
print("✅ 드라이브 마운트 완료")

## 섹션 1. 설정값 입력
> **여기만 수정하면 됩니다.** 아래 변수들을 실제 경로와 원하는 값으로 바꾸세요.

In [ ]:
# ── [수정 필요] 경로 설정 ────────────────────────────────────
import os
from pathlib import Path

# 학습 데이터 JSONL 파일 경로 (드라이브 내)
JSONL_PATH = "/content/drive/MyDrive/finetune/Legal_Agent_finetune_Dataset_Final.jsonl"

# 결과물 저장 루트 (드라이브)
OUTPUT_ROOT = Path("/content/drive/MyDrive/finetune/output")

# ── [수정 가능] 모델 및 학습 설정 ────────────────────────────
MODEL_ID       = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 4096    # OOM 시 2048로 낮추기
LOAD_IN_4BIT   = True

# LoRA 설정
LORA_RANK      = 32   # 법률 도메인 특화 — r=16보다 높은 표현력 필요
LORA_ALPHA     = 32   # 권장: alpha == r

# 학습/검증 분할
EVAL_RATIO     = 0.05    # 전체의 5%를 검증용으로 분리 (예: 765개 → train 727 / eval 38)
EVAL_STEPS     = 20      # 몇 스텝마다 검증 실행할지

# 학습 설정
BATCH_SIZE     = 1
GRAD_ACCUM     = 4
LEARNING_RATE  = 2e-4
MAX_STEPS      = None    # None이면 NUM_EPOCHS 사용
NUM_EPOCHS     = 3
WARMUP_STEPS   = 10
SEED           = 3407

# ── 출력 경로 자동 생성 ───────────────────────────────────────
from datetime import datetime
RUN_ID = datetime.now().strftime("%m%d_%H%M")

ADAPTER_DIR    = OUTPUT_ROOT / f"adapter_{RUN_ID}"
CHECKPOINT_DIR = OUTPUT_ROOT / f"checkpoints_{RUN_ID}"

for d in [ADAPTER_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Run ID       : {RUN_ID}")
print(f"학습 데이터  : {JSONL_PATH}")
print(f"어댑터 저장  : {ADAPTER_DIR}")
print(f"체크포인트   : {CHECKPOINT_DIR}")
print(f"검증 비율    : {EVAL_RATIO * 100:.0f}%")


## 섹션 2. 데이터 로드 · 포맷 변환 · Train/Val 분할

**순서**: JSONL 로드 → `<|im_start|>` 파싱 → `conversations` 구조 변환 → `apply_chat_template` 재포맷 → 셔플 → 분할

데이터셋의 `text` 필드는 `<|im_start|>` (ChatML) 형식이지만, Gemma-4 토크나이저는 `<|turn>user` / `<|turn>model` 형식을 사용합니다.  
`train_on_responses_only`가 올바르게 동작하려면 반드시 `apply_chat_template`으로 재포맷해야 합니다.

In [ ]:
import json, re
from datasets import Dataset

# ── JSONL 로드 ────────────────────────────────────────────────
records = []
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"총 샘플 수: {len(records)}")
print()
print("=== 원본 text 필드 앞 200자 (ChatML 포맷) ===")
print(records[0]["text"][:200])

# ── <|im_start|> → conversations 구조 파싱 ────────────────────
# Gemma-4 토크나이저는 <|turn>user/<|turn>model 형식을 사용
# 원본 ChatML(<|im_start|>)을 conversations로 변환 후
# apply_chat_template으로 올바른 Gemma-4 토큰 형식으로 재포맷해야
# train_on_responses_only가 <|turn>model 마커를 정확히 찾을 수 있습니다

ROLE_MAP = {"system": "system", "user": "user", "assistant": "assistant"}

def parse_im_start(text):
    """<|im_start|>role\ncontent<|im_end|> → [{role, content}, ...]"""
    pattern = re.compile(
        r"<\|im_start\|>(\w+)\n(.*?)<\|im_end\|>",
        re.DOTALL
    )
    return [
        {"role": ROLE_MAP.get(m.group(1), m.group(1)),
         "content": m.group(2).strip()}
        for m in pattern.finditer(text)
    ]

conversations = [parse_im_start(r["text"]) for r in records]

# 파싱 결과 확인
print(f"\n=== 파싱 결과 (샘플 0) ===")
for turn in conversations[0]:
    print(f"[{turn['role']}] {turn['content'][:80]}...")

## 섹션 3. 모델 로드 및 LoRA 설정

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name                = MODEL_ID,
    dtype                     = None,            # 자동 감지 (bf16 권장)
    max_seq_length            = MAX_SEQ_LENGTH,
    load_in_4bit              = LOAD_IN_4BIT,
    full_finetuning           = False,
    use_gradient_checkpointing = "unsloth",      # VRAM 절약 + 긴 컨텍스트 지원 (공식 권장)
    # token = "HF_TOKEN",  # private 모델일 경우 HuggingFace 토큰
)
print(f"✅ 모델 로드 완료: {MODEL_ID}")

In [ ]:
# LoRA 어댑터 설정
# E2B는 텍스트 전용 파인튜닝이므로 vision 레이어는 끔
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # 텍스트 전용
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r                          = LORA_RANK,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = 0,
    bias                       = "none",
    random_state               = SEED,
)
model.print_trainable_parameters()

In [ ]:
# 채팅 템플릿 설정 — 필수
# get_chat_template은 토크나이저가 <|turn> 특수 토큰을 올바르게 처리하도록 설정합니다.
# 데이터셋 포맷(im_start)과 무관하게 반드시 적용해야 train_on_responses_only가 정상 동작합니다.
#
# gemma-4        : 일반 응답 (thinking 태그 없음)
# gemma-4-thinking: <thought> 태그 포함 응답 — 데이터셋에 <thought>가 있으므로 이것을 사용
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4-thinking",  # 데이터셋에 <thought> 태그 포함
)
print("✅ 채팅 템플릿: gemma-4-thinking")
print("   토크나이저가 <|turn> 특수 토큰을 올바르게 인식합니다.")


In [ ]:
# ── 실행 순서 안내 ──────────────────────────────────────────
# 섹션 3 (모델 로드 → LoRA → 채팅 템플릿) 완료 후
# 섹션 2의 두 번째 셀(formatting_prompts_func)을 실행하세요.
# tokenizer가 먼저 로드되어 있어야 apply_chat_template이 동작합니다.
print("✅ 모델 및 LoRA 설정 완료")
print()
print("다음 단계:")
print("  → 섹션 2의 두 번째 셀로 돌아가서 실행 (데이터 포맷 변환 + 분할)")
print("  → 이후 섹션 4 (학습 실행) 진행")


In [ ]:
# ── apply_chat_template 적용 → Dataset 생성 ─────────────────
# tokenizer가 로드된 이후에 이 셀을 실행하세요 (섹션 3 실행 후)
# 재포맷 후 text 필드는 <|turn>user / <|turn>model 형식이 됩니다

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize              = False,
            add_generation_prompt = False,
        ).removeprefix("<bos>")
        for convo in examples["conversations"]
    ]
    return {"text": texts}

raw_dataset = Dataset.from_list([{"conversations": c} for c in conversations])
dataset     = raw_dataset.map(formatting_prompts_func, batched=True)

print("=== 재포맷 결과 (샘플 0 앞 300자) ===")
print(dataset[0]["text"][:300])
print()
print("✅ <|turn>user / <|turn>model 형식으로 변환 완료")
print("   train_on_responses_only가 <|turn>model 마커를 정확히 찾을 수 있습니다.")
print()

# ── Shuffle → Train / Eval 분할 ────────────────────────────────
dataset_shuffled = dataset.shuffle(seed=SEED)

eval_size  = max(1, int(len(dataset_shuffled) * EVAL_RATIO))
train_size = len(dataset_shuffled) - eval_size

train_dataset = dataset_shuffled.select(range(train_size))
eval_dataset  = dataset_shuffled.select(range(train_size, len(dataset_shuffled)))

print(f"전체 샘플  : {len(dataset_shuffled)}개")
print(f"학습 데이터: {len(train_dataset)}개")
print(f"검증 데이터: {len(eval_dataset)}개")
print()

# ── 판정 등급 분포 확인 ─────────────────────────────────────────
def count_verdicts(ds, name):
    counts = {'🔴': 0, '🟡': 0, '🟢': 0}
    for r in ds:
        for k in counts:
            if k in r['text']:
                counts[k] += 1
                break
    total = len(ds)
    print(f"{name}: 🔴 {counts['🔴']}({counts['🔴']/total*100:.0f}%) "
          f"🟡 {counts['🟡']}({counts['🟡']/total*100:.0f}%) "
          f"🟢 {counts['🟢']}({counts['🟢']/total*100:.0f}%)")

count_verdicts(train_dataset, 'Train')
count_verdicts(eval_dataset,  'Eval ')


## 섹션 4. 학습 실행

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = eval_dataset,   # ← 검증 데이터 연결
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = WARMUP_STEPS,
        num_train_epochs            = NUM_EPOCHS if MAX_STEPS is None else 1,
        max_steps                   = MAX_STEPS,
        learning_rate               = LEARNING_RATE,
        logging_steps               = 10,
        # ── 검증 설정 ──────────────────────────────────────────
        eval_strategy               = "steps",
        eval_steps                  = EVAL_STEPS,
        per_device_eval_batch_size  = 1,
        load_best_model_at_end      = True,   # 검증 loss 최저 시점 모델 자동 선택
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        # ── 체크포인트 설정 ────────────────────────────────────
        save_strategy               = "steps",
        save_steps                  = EVAL_STEPS,  # eval과 동일 주기로 저장
        save_total_limit            = 3,
        output_dir                  = str(CHECKPOINT_DIR),
        # ── 기타 ──────────────────────────────────────────────
        optim                       = "adamw_8bit",
        weight_decay                = 0.001,
        lr_scheduler_type           = "linear",
        seed                        = SEED,
        report_to                   = "none",
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        max_seq_length              = MAX_SEQ_LENGTH,
    ),
)

# ── assistant 응답 부분만 학습 ───────────────────────────────
# Gemma-4 실제 토큰 형식: <|turn>user / <|turn>model
# (데이터셋이 <|im_start|> 형식이더라도 토크나이저가 내부적으로 <|turn>으로 처리)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part    = "<|turn>model\n",
)
print("✅ Trainer 설정 완료")
print(f"   학습: {len(train_dataset)}개 / 검증: {len(eval_dataset)}개")
print(f"   검증 주기: {EVAL_STEPS} 스텝마다")
print(f"   최적 모델: eval_loss 최저 시점 자동 저장")
print(f"   응답 마스킹: <|turn>model 이후만 학습 (system/user 제외)")


In [ ]:
# ── 마스킹 검증 ──────────────────────────────────────────────
# user 입력이 -100으로 마스킹되고 assistant 응답만 남아있는지 확인
sample_ids    = trainer.train_dataset[0]["input_ids"]
sample_labels = trainer.train_dataset[0]["labels"]

decoded_input  = tokenizer.decode(sample_ids)
decoded_labels = tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in sample_labels]
).replace(tokenizer.pad_token, " ")

print("=== 입력 (앞 200자) ===")
print(decoded_input[:200])
print("\n=== 학습 대상 (assistant 응답만, 앞 200자) ===")
print(decoded_labels[:200])

In [ ]:
# ── 학습 실행 ────────────────────────────────────────────────
# E2B loss는 13~15가 정상입니다 (multimodal 모델 특성 — Unsloth 공식 확인)
print("학습 시작...")
trainer_stats = trainer.train()

print(f"\n✅ 학습 완료")
print(f"   학습 시간: {trainer_stats.metrics['train_runtime']:.0f}초")
print(f"   최종 loss: {trainer_stats.metrics['train_loss']:.4f}")
print("   ※ E2B loss 13~15는 정상입니다 (Unsloth 공식 문서 확인)")

## 섹션 5. 저장

### 방식 비교
| | 5A: PyTorch 어댑터 (권장) | 5B: 병합 GGUF |
|---|---|---|
| 결과물 | adapter_model.safetensors | 단일 GGUF 파일 |
| 서비스 구조 | 베이스 + 어댑터 ON/OFF | 무역 용도 전용 단일 파일 |
| OCR 등 베이스 용도 | 같은 프로세스에서 어댑터 OFF로 전환 가능 | 별도 베이스 모델 프로세스 필요 |
| 적합한 경우 | **베이스(OCR 등)와 무역 용도를 하나의 모델로** | 무역 전용 독립 배포 |
| 권장 여부 | ✅ 현재 운용 구조에 적합 | 전용 배포 시만 사용 |

> 어댑터는 무역 법리 하나만 운용합니다.  
> OCR 등 다른 기능은 어댑터 없이 베이스 모델 그대로 사용합니다.


In [ ]:
# ── 5A: PyTorch 어댑터 저장 (권장) ─────────────────────────
# LoRA 어댑터만 저장 — 베이스 모델은 그대로 유지
# 서비스 시 베이스 모델 로드 후 어댑터를 ON/OFF하여 용도 전환
#   어댑터 ON  → 무역 법리 판단
#   어댑터 OFF → OCR, 일반 용도 (베이스 모델 그대로)

# load_best_model_at_end=True로 인해 현재 model은 이미 최적 체크포인트 상태
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

adapter_files = list(ADAPTER_DIR.iterdir())
print(f"✅ 어댑터 저장 완료: {ADAPTER_DIR}")
print(f"   저장된 파일:")
for f in sorted(adapter_files):
    size_mb = f.stat().st_size / 1e6
    print(f"   - {f.name} ({size_mb:.1f} MB)")
print()
print("서비스 로드 예시:")
print("  model, tokenizer = FastModel.from_pretrained('unsloth/gemma-4-E2B-it', ...)")
print(f"  model.load_adapter('{ADAPTER_DIR}')  # 무역 법리 모드")
print("  model.disable_adapters()              # 베이스 모드 (OCR 등)")
print("  model.enable_adapters()               # 다시 무역 법리 모드")


In [ ]:
# ── 5B: 병합 GGUF 저장 (단일 배포 시) ──────────────────────
# 베이스 모델과 어댑터를 병합하여 단일 GGUF로 변환
# ※ 병합 후에는 어댑터 교체 불가 — 역할별 어댑터 구조에는 부적합
# ※ OCR 등 다른 어댑터와 병행 사용하지 않는 경우에만 사용

print("병합 GGUF는 어댑터 교체 구조에 적합하지 않습니다.")
print("단일 용도 배포가 필요한 경우 아래 주석을 해제하세요.")
print()

# GGUF_DIR = OUTPUT_ROOT / f"gguf_{RUN_ID}"
# GGUF_DIR.mkdir(parents=True, exist_ok=True)
#
# model.save_pretrained_gguf(
#     str(GGUF_DIR),
#     tokenizer,
#     quantization_method = "q4_k_m",
# )
# print(f"✅ 병합 GGUF 저장: {GGUF_DIR}")


In [ ]:
# ── 5C: llama.cpp convert_lora_to_gguf.py 방식 (선택) ────────
# 제시된 방식 — 원본 GGUF + 어댑터 GGUF를 서비스 시 함께 로드해야 함
# 5A로 충분하다면 이 셀은 실행하지 않아도 됩니다

print("llama.cpp 방식은 5A (Unsloth GGUF)로 대체 가능합니다.")
print("이 방식을 사용하려면 아래 주석을 해제하세요.")
print()

# import subprocess
# # llama.cpp 설치
# subprocess.run(["git", "clone", "https://github.com/ggerganov/llama.cpp"], check=True)
# subprocess.run(["pip", "install", "-r", "llama.cpp/requirements.txt", "-q"], check=True)
#
# # 어댑터 → GGUF 변환
# adapter_gguf = GGUF_DIR / "my_adapter.gguf"
# subprocess.run([
#     "python", "llama.cpp/convert_lora_to_gguf.py",
#     str(ADAPTER_DIR),
#     "--outfile", str(adapter_gguf),
# ], check=True)
#
# print(f"✅ 어댑터 GGUF 저장: {adapter_gguf}")
# print("   ※ 서비스 시 원본 GGUF와 이 어댑터 GGUF를 함께 로드해야 합니다.")

## 섹션 6. 샘플 추론 검증

In [ ]:
# 학습된 모델로 샘플 추론
# 학습 데이터의 system 프롬프트를 직접 사용
import re

# JSONL에서 system 내용 추출 (첫 번째 샘플)
first_text = records[0]["text"]
system_match = re.search(r"<\|im_start\|>system\n(.*?)\n<\|im_end\|>", first_text, re.DOTALL)
sample_system = system_match.group(1) if system_match else "당신은 국제 무역 법률 전문가입니다."

test_input = """[지시사항]\n아래 계약서 초안의 법적 리스크를 분석하라.\n\n[입력 데이터]\n【계약 개요】\n- 준거법: CISG\n- 계약 당사자: 한국 매도인 ↔ 영국 매수인\n- 품목: 냉동 명태 필렛 (HS 0304.71)\n- 계약 금액: USD 48,000\n- 인도 조건: CIF Liverpool\n\n【계약서 초안 주요 조항】\n- 보험 조항: 매도인이 부보, 부보율 100%, ICC(C) 조건\n- 준거법·분쟁해결: English law, ICC 중재"""

messages = [
    {"role": "system",    "content": sample_system},
    {"role": "user",      "content": test_input},
]

from transformers import TextStreamer

from unsloth import FastModel
FastModel.for_inference(model)  # 추론 모드 전환

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize              = True,
    add_generation_prompt = True,
    return_tensors        = "pt",
    return_dict           = True,
).to("cuda")

print("=== 추론 결과 ===")
_ = model.generate(
    **inputs,
    max_new_tokens = 2048,  # 법리 보고서 전체 출력 보장 (512는 중간에 끊김)
    use_cache      = True,
    temperature    = 1.0,
    top_p          = 0.95,
    top_k          = 64,
    streamer       = TextStreamer(tokenizer, skip_prompt=True),
)

In [ ]:
# ── 저장 경로 최종 확인 ─────────────────────────────────────
print("=" * 50)
print("저장 완료 경로 요약")
print("=" * 50)
print(f"PyTorch 어댑터 : {ADAPTER_DIR}")
print(f"체크포인트     : {CHECKPOINT_DIR}")
print()
print("서비스 로드 예시 (어댑터 교체 방식):")
print("  model, tok = FastModel.from_pretrained('unsloth/gemma-4-E2B-it', ...)")
print(f"  model.load_adapter('{ADAPTER_DIR}')  # 무역 법리 어댑터")
print("  model.disable_adapters()              # 베이스 모델로 복귀")
print()
print(f"드라이브에서 확인: {OUTPUT_ROOT}")